# Ensemble Analyzer
Discovers all experiment artifacts, checks OOF / pred alignment, and evaluates whether ensembling will help.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from itertools import combinations
from scipy.optimize import minimize
from sklearn.metrics import log_loss, balanced_accuracy_score
from scipy.stats import spearmanr

from config import ART_DIR, features

# ── Resolve artifact root ─────────────────────────────────────────────────────
ART_ROOT = Path(ART_DIR)   # e.g. project/artifacts/
TARGET   = features.target
print(f"Scanning: {ART_ROOT}")

## 1  Discover experiments

In [ ]:
def discover_experiments(art_root: Path) -> pd.DataFrame:
    """Walk artifact subfolders and load meta.json for every experiment."""
    rows = []
    for exp_dir in sorted(art_root.iterdir()):
        if not exp_dir.is_dir():
            continue
        meta_files = list(exp_dir.glob("*_meta.json"))
        oof_files  = list(exp_dir.glob("*_oof.npy"))
        pred_files = list(exp_dir.glob("*_preds.npy"))
        if not (meta_files and oof_files and pred_files):
            continue
        meta = json.loads(meta_files[0].read_text())
        rows.append({
            "exp_name":   exp_dir.name,
            "exp_dir":    exp_dir,
            "oof_path":   oof_files[0],
            "pred_path":  pred_files[0],
            "cv_metric":  meta.get("cv_auc",     meta.get("mean_cv_metric", np.nan)),
            "cv_logloss": meta.get("cv_logloss", meta.get("mean_logloss",   np.nan)),
            "train_file": meta.get("train",      ""),
            "created":    meta.get("created",    ""),
        })
    return pd.DataFrame(rows).sort_values("cv_metric", ascending=False).reset_index(drop=True)

df_exps = discover_experiments(ART_ROOT)
print(f"Found {len(df_exps)} experiments")
df_exps[["exp_name", "cv_metric", "cv_logloss", "created", "train_file"]]

## 2  Select experiments to ensemble
Edit `SELECTED` to pick a subset, or leave as `None` to use all.

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────────────────────
SELECTED = None   # e.g. ["xgb1_base", "lgb1_base", "cat1_base"]  or None = all
# ─────────────────────────────────────────────────────────────────────────────

if SELECTED is not None:
    df_sel = df_exps[df_exps["exp_name"].isin(SELECTED)].reset_index(drop=True)
else:
    df_sel = df_exps.copy()

assert len(df_sel) >= 2, "Need at least 2 experiments to analyse ensemble."

# Load OOF / pred arrays keyed by exp_name
oofs  = {row.exp_name: np.load(row.oof_path)  for row in df_sel.itertuples()}
preds = {row.exp_name: np.load(row.pred_path) for row in df_sel.itertuples()}

n_classes = next(iter(oofs.values())).shape[1]
n_train   = next(iter(oofs.values())).shape[0]
print(f"n_train={n_train}, n_classes={n_classes}")

# Validate shapes match
for name, oof in oofs.items():
    assert oof.shape == (n_train, n_classes), f"{name}: OOF shape mismatch {oof.shape}"
for name, pred in preds.items():
    assert pred.shape[1] == n_classes, f"{name}: pred shape mismatch {pred.shape}"

print("All shapes consistent ✓")
df_sel[["exp_name", "cv_metric", "cv_logloss"]]

## 3  Load ground-truth OOF labels
Requires the training parquet used by these experiments (reads `TARGET` column).

In [ ]:
from config import PROC_DIR

# Auto-detect the training file from the first experiment's meta
train_file = df_sel.iloc[0]["train_file"]
train_path = Path(PROC_DIR) / train_file

df_train = pd.read_parquet(train_path)
y_raw    = df_train[TARGET].cat.codes.values      # integer class codes
classes  = df_train[TARGET].cat.categories.tolist()

assert len(y_raw) == n_train, f"Label length {len(y_raw)} != OOF length {n_train}"
print(f"Labels loaded: {len(y_raw)} rows, classes={classes}")

# Recompute log-loss from OOF to cross-check against meta.json
for name, oof in oofs.items():
    ll = log_loss(y_raw, oof)
    meta_ll = df_sel.loc[df_sel.exp_name == name, "cv_logloss"].values[0]
    match = "✓" if abs(ll - meta_ll) < 1e-3 else f"⚠ meta={meta_ll:.5f}"
    print(f"  {name:<30} OOF logloss={ll:.5f}  {match}")

## 4  OOF prediction correlation
Low inter-model correlation on OOF probabilities is the strongest signal that ensembling will help.

In [ ]:
names = list(oofs.keys())

# ── Pearson on max-class OOF probability ─────────────────────────────────────
oof_maxprob = {n: oofs[n].max(axis=1) for n in names}   # shape (n_train,)

corr_matrix = pd.DataFrame(index=names, columns=names, dtype=float)
spear_matrix = pd.DataFrame(index=names, columns=names, dtype=float)
for a, b in [(a, b) for a in names for b in names]:
    r_p = np.corrcoef(oof_maxprob[a], oof_maxprob[b])[0, 1]
    r_s, _ = spearmanr(oof_maxprob[a], oof_maxprob[b])
    corr_matrix.loc[a, b]  = r_p
    spear_matrix.loc[a, b] = r_s

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, mat, title in zip(axes,
                          [corr_matrix, spear_matrix],
                          ["Pearson r (OOF max-prob)", "Spearman r (OOF max-prob)"]):
    sns.heatmap(mat.astype(float), annot=True, fmt=".3f",
                cmap="RdYlGn_r", vmin=0.8, vmax=1.0, ax=ax,
                linewidths=0.5, square=True)
    ax.set_title(title)

plt.suptitle("OOF inter-model correlation  (lower = better diversity)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(ART_ROOT / "ensemble_oof_corr.png", dpi=150, bbox_inches="tight")
plt.show()

# Verdict
for a, b in combinations(names, 2):
    r = corr_matrix.loc[a, b]
    verdict = "✓ diverse" if r < 0.96 else ("~ marginal" if r < 0.99 else "✗ redundant")
    print(f"  {a:<28} × {b:<28}  Pearson={r:.4f}  {verdict}")

## 5  OOF disagreement analysis
Where do models disagree? Rows with high disagreement are either noisy or hard cases.

In [ ]:
# Per-row std of predicted max-probability across models
oof_stack = np.stack([oofs[n].max(axis=1) for n in names], axis=1)  # (n_train, n_models)
row_std    = oof_stack.std(axis=1)

# Argmax disagreement: how often do models predict a different class?
argmax_stack = np.stack([np.argmax(oofs[n], axis=1) for n in names], axis=1)  # (n_train, n_models)
disagreement = (argmax_stack != argmax_stack[:, [0]]).any(axis=1)  # True where any model differs from model 0

print(f"Rows where ≥1 model disagrees with {names[0]}: "
      f"{disagreement.sum()} / {n_train}  "
      f"({100*disagreement.mean():.1f}%)")

# Distribution of row-level std
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(row_std, bins=60, color="steelblue", edgecolor="white", linewidth=0.4)
axes[0].set_xlabel("Std of max-prob across models (per row)")
axes[0].set_ylabel("Count")
axes[0].set_title("OOF Probability Spread Distribution")
axes[0].axvline(row_std.mean(), color="crimson", linestyle="--", label=f"mean={row_std.mean():.4f}")
axes[0].legend()

# Disagreement by true class
class_disagree = pd.Series(disagreement).groupby(y_raw).mean().rename(index=dict(enumerate(classes)))
class_disagree.plot.bar(ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("Disagreement Rate by True Class")
axes[1].set_xlabel("True class")
axes[1].set_ylabel("Fraction of rows with disagreement")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(ART_ROOT / "ensemble_disagreement.png", dpi=150, bbox_inches="tight")
plt.show()

## 6  Ensemble strategies — OOF benchmark
Compare equal-weight average, rank average, best-single, and (optionally) optimised weights.

In [ ]:
def oof_logloss(proba: np.ndarray) -> float:
    return log_loss(y_raw, proba)

def oof_bacc(proba: np.ndarray) -> float:
    return balanced_accuracy_score(y_raw, np.argmax(proba, axis=1))

results = []

# Individual models
for name in names:
    results.append({
        "strategy":   name,
        "oof_logloss": oof_logloss(oofs[name]),
        "oof_bacc":    oof_bacc(oofs[name]),
        "note":        "single model",
    })

# Equal-weight average
oof_avg = np.mean([oofs[n] for n in names], axis=0)
results.append({
    "strategy":   "equal_avg",
    "oof_logloss": oof_logloss(oof_avg),
    "oof_bacc":    oof_bacc(oof_avg),
    "note":        f"avg of {len(names)} models",
})

# Rank-based average (converts each model's max-prob to a rank before averaging)
from scipy.stats import rankdata
ranked = []
for n in names:
    m = oofs[n].copy()
    for c in range(n_classes):
        m[:, c] = rankdata(m[:, c]) / n_train
    ranked.append(m)
oof_rank = np.mean(ranked, axis=0)
oof_rank = oof_rank / oof_rank.sum(axis=1, keepdims=True)  # renormalise
results.append({
    "strategy":   "rank_avg",
    "oof_logloss": oof_logloss(oof_rank),
    "oof_bacc":    oof_bacc(oof_rank),
    "note":        "rank-transformed avg",
})

# Weighted average — weights optimised on OOF logloss via Nelder-Mead
def blended_ll(w_raw):
    w = np.exp(w_raw) / np.exp(w_raw).sum()          # softmax → positive sum-to-1
    blended = sum(w[i] * oofs[n] for i, n in enumerate(names))
    return log_loss(y_raw, blended)

opt = minimize(blended_ll, x0=np.zeros(len(names)), method="Nelder-Mead",
               options={"maxiter": 5000, "xatol": 1e-6})
w_opt = np.exp(opt.x) / np.exp(opt.x).sum()
oof_opt = sum(w_opt[i] * oofs[n] for i, n in enumerate(names))

results.append({
    "strategy":   "opt_weights",
    "oof_logloss": oof_logloss(oof_opt),
    "oof_bacc":    oof_bacc(oof_opt),
    "note":        "  ".join(f"{n}={w:.3f}" for n, w in zip(names, w_opt)),
})

df_results = pd.DataFrame(results).sort_values("oof_logloss")
df_results

## 7  Visualise strategy comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = ["#e07b54" if "single" in r else "#4a90d9" for r in df_results["note"]]

df_results.plot.barh(x="strategy", y="oof_logloss", ax=axes[0],
                     color=palette, legend=False, edgecolor="white")
axes[0].set_title("OOF Log-Loss by Strategy  (lower = better)")
axes[0].set_xlabel("log-loss")
axes[0].invert_yaxis()

# Annotate improvement vs best single
best_single_ll = df_results[df_results.note == "single model"]["oof_logloss"].min()
for patch, (_, row) in zip(axes[0].patches, df_results.iterrows()):
    delta = row["oof_logloss"] - best_single_ll
    label = f"{delta:+.5f}" if row["note"] != "single model" else ""
    axes[0].text(patch.get_width() + 0.0002, patch.get_y() + patch.get_height() / 2,
                 label, va="center", fontsize=8, color="#333")

df_results.plot.barh(x="strategy", y="oof_bacc", ax=axes[1],
                     color=palette, legend=False, edgecolor="white")
axes[1].set_title("OOF Balanced Accuracy  (higher = better)")
axes[1].set_xlabel("balanced accuracy")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(ART_ROOT / "ensemble_strategy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8  OOF calibration check
Poorly calibrated models hurt ensembles. Check if predicted probabilities match actual class frequencies.

In [ ]:
from sklearn.calibration import calibration_curve

# One vs rest calibration for class 0 (adapt as needed)
fig, axes = plt.subplots(1, len(names), figsize=(5 * len(names), 5), sharey=True)
if len(names) == 1:
    axes = [axes]

y_bin = (y_raw == 0).astype(int)   # binary: class 0 vs rest

for ax, name in zip(axes, names):
    prob_pos = oofs[name][:, 0]
    frac_pos, mean_pred = calibration_curve(y_bin, prob_pos, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, "s-", label=name, linewidth=2)
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect")
    ax.set_title(f"{name}\nCalibration (class 0 vs rest)")
    ax.set_xlabel("Mean predicted prob")
    ax.set_ylabel("Fraction of positives")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(ART_ROOT / "ensemble_calibration.png", dpi=150, bbox_inches="tight")
plt.show()

## 9  Pred distribution alignment
Checks that test-time predictions are consistent with the OOF distribution — a mismatch signals CV / test domain shift.

In [ ]:
n_cols = n_classes
fig, axes = plt.subplots(len(names), n_cols,
                         figsize=(4 * n_cols, 3 * len(names)),
                         squeeze=False)

for row_i, name in enumerate(names):
    for c in range(n_cols):
        ax = axes[row_i][c]
        oof_c  = oofs[name][:, c]
        pred_c = preds[name][:, c]

        ax.hist(oof_c,  bins=50, alpha=0.55, density=True, color="steelblue", label="OOF")
        ax.hist(pred_c, bins=50, alpha=0.55, density=True, color="coral",     label="Test pred")

        ks_stat = np.abs(np.sort(oof_c) - np.sort(np.interp(
            np.linspace(0, 1, len(oof_c)),
            np.linspace(0, 1, len(pred_c)),
            np.sort(pred_c)))).max()

        ax.set_title(f"{name} | class {c} (KS≈{ks_stat:.3f})", fontsize=9)
        if row_i == 0:
            ax.legend(fontsize=8)

plt.suptitle("OOF vs Test Pred Distributions  (should overlap closely)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(ART_ROOT / "ensemble_pred_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 10  Test pred correlation
Cross-check that models that are diverse on OOF are also diverse on test preds. If OOF diversity is high but test preds are highly correlated, something is off.

In [ ]:
pred_maxprob = {n: preds[n].max(axis=1) for n in names}

pred_corr = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        pred_corr.loc[a, b] = np.corrcoef(pred_maxprob[a], pred_maxprob[b])[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(corr_matrix.astype(float), annot=True, fmt=".3f",
            cmap="RdYlGn_r", vmin=0.8, vmax=1.0, ax=axes[0],
            linewidths=0.5, square=True)
axes[0].set_title("OOF Pearson Correlation")

sns.heatmap(pred_corr.astype(float), annot=True, fmt=".3f",
            cmap="RdYlGn_r", vmin=0.8, vmax=1.0, ax=axes[1],
            linewidths=0.5, square=True)
axes[1].set_title("Test Pred Pearson Correlation")

plt.suptitle("OOF vs Test-pred correlation  (should be similar)", fontsize=12)
plt.tight_layout()
plt.savefig(ART_ROOT / "ensemble_oof_vs_pred_corr.png", dpi=150, bbox_inches="tight")
plt.show()

# Flag suspicious divergence
for a, b in combinations(names, 2):
    oof_r  = corr_matrix.loc[a, b]
    pred_r = pred_corr.loc[a, b]
    delta  = abs(float(pred_r) - float(oof_r))
    flag   = "⚠ investigate" if delta > 0.03 else "ok"
    print(f"  {a:<25} × {b:<25}  OOF_r={oof_r:.3f}  pred_r={pred_r:.3f}  Δ={delta:.3f}  {flag}")

## 11  Build best ensemble pred file
Applies the best-performing strategy from Section 6 to the test predictions and saves a submission-ready CSV.

In [ ]:
from config import RAW_DIR, PROC_DIR, SUB_DIR, comp_cfg

# ── USER CONFIG ───────────────────────────────────────────────────────────────
BEST_STRATEGY = "opt_weights"   # "equal_avg" | "rank_avg" | "opt_weights"
ENSEMBLE_NAME = "ensemble_v1"   # name for the submission file
LABEL_FLAG    = True            # True → decode to class label strings
# ─────────────────────────────────────────────────────────────────────────────

if BEST_STRATEGY == "equal_avg":
    final_pred = np.mean([preds[n] for n in names], axis=0)
elif BEST_STRATEGY == "rank_avg":
    ranked_test = []
    for n in names:
        m = preds[n].copy()
        for c in range(n_classes):
            m[:, c] = rankdata(m[:, c]) / preds[n].shape[0]
        ranked_test.append(m)
    final_pred = np.mean(ranked_test, axis=0)
    final_pred = final_pred / final_pred.sum(axis=1, keepdims=True)
elif BEST_STRATEGY == "opt_weights":
    # w_opt already computed in Section 6
    final_pred = sum(w_opt[i] * preds[n] for i, n in enumerate(names))
else:
    raise ValueError(f"Unknown strategy: {BEST_STRATEGY}")

# Decode to labels and save
df_sub  = pd.read_csv(comp_cfg.sample_sub_path(RAW_DIR))
pred_ord = np.argmax(final_pred, axis=1)
df_sub[TARGET] = pd.Categorical.from_codes(codes=pred_ord, categories=classes)

if LABEL_FLAG:
    df_sub[TARGET] = df_sub[TARGET].astype(str)
else:
    df_sub[TARGET] = df_sub[TARGET].cat.codes.astype("int8")

sub_path = Path(SUB_DIR) / f"{ENSEMBLE_NAME}.csv"
df_sub.to_csv(sub_path, index=False)

# Also save the raw probability array for downstream stacking
np.save(ART_ROOT / f"{ENSEMBLE_NAME}_preds.npy", final_pred)

# Save ensemble metadata
ensemble_meta = {
    "name":          ENSEMBLE_NAME,
    "strategy":      BEST_STRATEGY,
    "models":        names,
    "weights":       w_opt.tolist() if BEST_STRATEGY == "opt_weights" else None,
    "oof_logloss":   float(df_results[df_results.strategy == BEST_STRATEGY]["oof_logloss"].values[0]),
    "oof_bacc":      float(df_results[df_results.strategy == BEST_STRATEGY]["oof_bacc"].values[0]),
}
(ART_ROOT / f"{ENSEMBLE_NAME}_meta.json").write_text(json.dumps(ensemble_meta, indent=2))

print(f"Submission saved → {sub_path}")
print(json.dumps(ensemble_meta, indent=2))

## 12  Summary verdict

In [ ]:
best_single = df_results[df_results.note == "single model"].sort_values("oof_logloss").iloc[0]
best_ens    = df_results[df_results.note != "single model"].sort_values("oof_logloss").iloc[0]

ll_improvement  = best_single["oof_logloss"] - best_ens["oof_logloss"]
bacc_improvement = best_ens["oof_bacc"] - best_single["oof_bacc"]

mean_pair_corr = float(np.mean([
    corr_matrix.loc[a, b] for a, b in combinations(names, 2)
]))

print("=" * 58)
print("ENSEMBLE VERDICT")
print("=" * 58)
print(f"  Best single model  : {best_single.strategy:<28} logloss={best_single.oof_logloss:.5f}")
print(f"  Best ensemble      : {best_ens.strategy:<28} logloss={best_ens.oof_logloss:.5f}")
print(f"  OOF logloss gain   : {ll_improvement:+.5f}")
print(f"  Balanced-acc gain  : {bacc_improvement:+.5f}")
print(f"  Mean pair OOF corr : {mean_pair_corr:.4f}")
print()

if ll_improvement > 0.002:
    print("✓  Ensembling is worthwhile — submit the ensemble.")
elif ll_improvement > 0:
    print("~  Marginal gain — ensemble if diversity grows with more models.")
else:
    print("✗  Ensemble does not improve on best single — models are too similar.")

if mean_pair_corr > 0.99:
    print("⚠  Models are near-identical — add a structurally different model (e.g. NN).")
elif mean_pair_corr > 0.97:
    print("~  Low diversity — different FE parquet or architecture may help more.")
else:
    print("✓  Healthy model diversity.")
print("=" * 58)